In [3]:
from langchain.tools import tool

@tool
def square_root(x:float) -> float:
  """claculate the square root of the number"""
  return x ** 0.5

@tool
def square(x:float) -> float:
  """Calculate the square of a number"""
  return x ** 2


In [2]:
!pip install langchain-openai

#making subagent

In [9]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm_model = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    model="openrouter/free",
    api_key = "add your api key"
)

sub_agent1 = create_agent(
    model = llm_model,
    tools =[square_root]
)

sub_agent2 = create_agent(
    model = llm_model,
    tools= [square]
)

#calling subagents

In [10]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x : float) -> float:
  """calculate the subagent q inorder to calculate the square_root"""
  response = sub_agent1.invoke({"messages": [HumanMessage(content= f"calculate the square_root of {x}")]})
  return response["messages"][-1].content

@tool
def call_subagent_2(x : float) -> float:
  """calculate the subagent q inorder to calculate the square_root"""
  response = sub_agent2.invoke({"messages": [HumanMessage(content= f"calculate the square of {x}")]})
  return response["messages"][-1].content

##creating main agent
main_agent = create_agent(
    model = llm_model,
    tools = [call_subagent_1 , call_subagent_2],
    system_prompt = "you are a helpfull assistant who can call subagents to calculate square root and square of a number"
)

In [11]:
question = "what is the square root of 25"
response = main_agent.invoke({"messages":[HumanMessage(content=question)]})

In [12]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='what is the square root of 25', additional_kwargs={}, response_metadata={}, id='29823a43-d36f-4f01-8013-eae96880c299'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 376, 'total_tokens': 475, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 66, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3.5-lightning:free', 'system_fingerprint': None, 'id': 'gen-1791056988-w43tZRDXNJzL3jhlVSXw', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a10350-ca2d-7311-bb